In [ ]:
from pathlib import Path
import sys
import numpy as np
import yaml
from PIL import Image, ImageDraw

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data.dataset import BaseDataset
from src.data.folder import expand

config = yaml.safe_load((ROOT / 'config/finetune_test.yaml').read_text())
paths, conds, labels = expand(config['data']['train']['folders'])
datasets = {
    prompt: BaseDataset(paths, prompts=[prompt], conds=conds, labels=labels)
    for prompt in ('point', 'box')
}
image_items = {}
for index, (sample_index, _) in enumerate(datasets['point'].items):
    image_id = datasets['point']._load_sample(sample_index).image.id
    image_items.setdefault(image_id, []).append(index)

def draw_item(item):
    image = Image.fromarray(item['image']).convert('RGB')
    mask = Image.fromarray((item['target'] * 255).astype('uint8')).resize(image.size)
    color = Image.new('RGB', image.size, (255, 40, 160))
    image = Image.composite(Image.blend(image, color, 0.35), image, mask)
    draw = ImageDraw.Draw(image)
    prompt = item['prompt']
    if prompt['points'] is not None:
        for x, y in prompt['points']:
            draw.ellipse((x - 10, y - 10, x + 10, y + 10), fill='cyan')
    if prompt['box'] is not None:
        draw.rectangle(tuple(prompt['box']), outline='cyan', width=5)
    draw.text((10, 10), f"{prompt['type']}  class={item['label_target'].astype(int).tolist()}", fill='white')
    return image.resize((504, 504))


In [ ]:
image_ids = np.random.choice(list(image_items), 2, replace=False)
indices = [np.random.choice(image_items[image_id]) for image_id in image_ids]
panels = [draw_item(datasets[prompt][index]) for index in indices for prompt in ('point', 'box')]
sheet = Image.new('RGB', (1008, 1008), 'white')
for panel_index, panel in enumerate(panels):
    sheet.paste(panel, ((panel_index % 2) * 504, (panel_index // 2) * 504))
sheet
